In [ ]:
#@title Copyright 2019 Google LLC. { display-mode: "form" }
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table class="ee-notebook-buttons" align="left"><td>
<a target="_blank"  href="http://colab.research.google.com/github/google/earthengine-community/blob/master/guides/linked/ee-api-colab-setup.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" /> Run in Google Colab</a>
</td><td>
<a target="_blank"  href="https://github.com/google/earthengine-community/blob/master/guides/linked/ee-api-colab-setup.ipynb"><img width=32px src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" /> View source on GitHub</a></td></table>

# Earth Engine Python API Colab Setup

This notebook demonstrates how to setup the Earth Engine Python API in Colab and provides several examples of how to print and visualize Earth Engine processed data.

## Import API and get credentials

The Earth Engine API is installed by default in Google Colaboratory so requires only importing and authenticating. These steps must be completed for each new Colab session, if you restart your Colab kernel, or if your Colab virtual machine is recycled due to inactivity.

### Import the API

Run the following cell to import the API into your session.

In [1]:
import ee

### Authenticate and initialize

Run the `ee.Authenticate` function to authenticate your access to Earth Engine servers and `ee.Initialize` to initialize it. Upon running the following cell you'll be asked to grant Earth Engine access to your Google account. Follow the instructions printed to the cell.

In [2]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='cogent-coyote-500016-n3')

In [3]:
# ============================ CONFIG ======================================
PROJECT      = "cogent-coyote-500016-n3"          # <-- CHANGE THIS to your EE Cloud project
INPUT_XLSX   = "C:\Users\syed.noor\OneDrive\ICDDR,B\BDHS\Climate and mental health\cluster_date_climate.xlsx"   # input file (this script's folder)
OUTPUT_CSV   = "era5land_60day_temp.csv"     # output file
WINDOW_DAYS  = 60                             # exposure window length (days preceding interview)
BATCH_SIZE   = 200                            # points per Earth Engine request (200 is safe)
DATE_COL     = "interview_date"
LAT_COL      = "latnum"
LON_COL      = "longnum"
CLUST_COL    = "dhsclust"
# ERA5-Land daily aggregated; band temperature_2m is daily MEAN 2m temp in KELVIN
EE_DATASET   = "ECMWF/ERA5_LAND/DAILY_AGGR"
EE_BAND      = "temperature_2m"
# ==========================================================================


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (3947668712.py, line 3)

In [ ]:


def init_ee():
    """Initialise Earth Engine; authenticate on first run if needed."""
    try:
        ee.Initialize(project=PROJECT)
    except Exception:
        print("Authenticating to Earth Engine (browser will open)...")
        ee.Authenticate()
        ee.Initialize(project=PROJECT)
    print("Earth Engine initialised on project:", PROJECT)


def build_feature(row):
    """Build an ee.Feature carrying the point geometry, the 60-day start/end, and an id."""
    end = ee.Date(str(pd.to_datetime(row[DATE_COL]).date()))          # interview date (exclusive upper bound handled below)
    start = end.advance(-WINDOW_DAYS, "day")                          # 60 days before
    geom = ee.Geometry.Point([float(row[LON_COL]), float(row[LAT_COL])])
    return ee.Feature(geom, {
        "row_id": int(row["row_id"]),
        "start": start.millis(),
        "end": end.millis(),
    })


def extract_batch(features):
    """
    For a batch of features, compute the 60-day mean temperature_2m at each point.
    Returns list of dicts: {row_id, era5land_t2m_60d (Celsius)}.
    """
    fc = ee.FeatureCollection(features)

    def per_feature(feat):
        feat = ee.Feature(feat)
        start = ee.Date(feat.get("start"))
        end = ee.Date(feat.get("end"))
        # ERA5-Land daily images over the 60-day window [start, end)
        col = (ee.ImageCollection(EE_DATASET)
               .select(EE_BAND)
               .filterDate(start, end))
        # mean over the time window, sampled at the point
        mean_img = col.mean()
        val = mean_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=feat.geometry(),
            scale=11132,            # ~0.1 deg native ERA5-Land resolution in metres
            maxPixels=1e9,
        ).get(EE_BAND)
        return feat.set("t2m_kelvin", val)

    out = fc.map(per_feature)
    # pull results to the client
    data = out.getInfo()["features"]
    results = []
    for f in data:
        props = f["properties"]
        kelvin = props.get("t2m_kelvin", None)
        celsius = (kelvin - 273.15) if kelvin is not None else None
        results.append({
            "row_id": props["row_id"],
            "era5land_t2m_60d": round(celsius, 4) if celsius is not None else None,
        })
    return results


def main():
    init_ee()

    # ---- load input ----
    df = pd.read_excel(INPUT_XLSX)
    df = df.reset_index(drop=True)
    df["row_id"] = df.index
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    n = len(df)
    print(f"Loaded {n} cluster-date rows.")

    # ---- build features ----
    feats = [build_feature(r) for _, r in df.iterrows()]

    # ---- process in batches ----
    all_results = []
    n_batches = (n + BATCH_SIZE - 1) // BATCH_SIZE
    for b in range(n_batches):
        lo = b * BATCH_SIZE
        hi = min((b + 1) * BATCH_SIZE, n)
        batch_feats = feats[lo:hi]
        ok = False
        for attempt in range(1, 6):          # up to 5 retries per batch
            try:
                res = extract_batch(batch_feats)
                all_results.extend(res)
                ok = True
                print(f"  batch {b+1}/{n_batches}  rows {lo}-{hi-1}  OK")
                break
            except Exception as e:
                wait = attempt * 5
                print(f"  batch {b+1}/{n_batches} attempt {attempt} failed: {e}. "
                      f"retry in {wait}s...")
                time.sleep(wait)
        if not ok:
            print(f"  !! batch {b+1} FAILED after retries. Saving progress and stopping.")
            break
        time.sleep(1)   # gentle pause between batches

    # ---- merge & save ----
    res_df = pd.DataFrame(all_results)
    merged = df.merge(res_df, on="row_id", how="left")
    merged = merged.drop(columns=["row_id"])
    merged.to_csv(OUTPUT_CSV, index=False)

    got = merged["era5land_t2m_60d"].notna().sum()
    print(f"\nDone. Extracted {got}/{n} values.")
    if got < n:
        print(f"WARNING: {n-got} rows missing - rerun to fill gaps if needed.")
    print(f"Saved -> {OUTPUT_CSV}")
    print("\nSummary of 60-day mean temperature (Celsius):")
    print(merged["era5land_t2m_60d"].describe())


if __name__ == "__main__":
    main()